In [6]:
import os
os.chdir("../../")

In [7]:
os.getcwd()

'/Users/supawitjunsiritrakhoon/Desktop/Customer_Churn_Prediction/Project_file/customer-churn-prediction'

In [8]:
import yaml
schema_path = "src/churn_prediction/config/data_validation/schemas/customer_profile.schema.yaml"
with open(schema_path, 'r', encoding='utf-8') as f:
    schema = yaml.safe_load(f)
schema

{'schema_version': '1.0',
 'columns': {'user_id': {'description': 'Unique identifier for each customer',
   'type': 'string',
   'data_type': 'VARCHAR(50)',
   'nullable': False,
   'unique_keys': True,
   'constraints': [{'type': 'primary_key',
     'description': 'Primary key constraint'},
    {'type': 'pattern',
     'value': '^[A-Za-z0-9_-]+$',
     'description': 'Alphanumeric, underscore, and hyphen only'}]},
  'first_name': {'description': "Customer's first name",
   'type': 'string',
   'data_type': 'VARCHAR(100)',
   'nullable': False,
   'constraints': [{'type': 'min_length', 'value': 1},
    {'type': 'max_length', 'value': 100},
    {'type': 'pattern',
     'value': "^[a-zA-Z\\s'-]+$",
     'description': 'Letters, spaces, apostrophes, and hyphens only'}]},
  'last_name': {'description': "Customer's last name",
   'type': 'string',
   'data_type': 'VARCHAR(100)',
   'nullable': False,
   'constraints': [{'type': 'min_length', 'value': 1},
    {'type': 'max_length', 'value': 

In [9]:
from pydantic import BaseModel, Field
from typing import Optional, List, Any, Dict

class ConstraintParameter(BaseModel):
    """
    Constraint Parameter Model

    Attributes:
        type (str): Type of constraint, e.g. primary_key, pattern, min_length
        value (Optional[Any]): Value of the constraint, if applicable
        description (Optional[str]): Description of the constraint
    """
    type: str = Field(..., description="Type of constraint, e.g. primary_key, pattern, min_length")
    value: Optional[Any] = Field(None, description="Value of the constraint, if applicable")
    description: Optional[str] = Field(None, description="Description of the constraint")


class ColumnParameter(BaseModel):
    """
    Column Parameter Model

    Attributes:
        description (str): Description of the column
        type (str): Logical type (e.g. string, integer)
        data_type (Optional[str]): Physical database type, e.g. VARCHAR(50)
        nullable (bool): Whether column allows NULL values
        unique (Optional[bool]): Whether column values must be unique
        constraints (Optional[List[ConstraintParameter]]): List of constraints for this column
    """
    description: str = Field(..., description="Description of the column")
    type: str = Field(..., description="Logical type (e.g. string, integer)")
    data_type: Optional[str] = Field(None, description="Physical database type, e.g. VARCHAR(50)")
    nullable: bool = Field(True, description="Whether column allows NULL values")
    unique: Optional[bool] = Field(None, description="Whether column values must be unique")
    constraints: Optional[List[ConstraintParameter]] = Field(
        default_factory=list,
        description="List of constraints for this column"
    )


class RuleParameter(BaseModel):
    """
    Rule Configuration Model

    Attributes:
        enabled (bool): Whether this rule is enabled
        description (Optional[str]): Description of the rule
    """
    enabled: bool = Field(..., description="Whether this rule is enabled")
    description: Optional[str] = Field(None, description="Description of the rule")


class QualityRulesParameter(BaseModel):
    """
    Quality Rules Parameter Model

    Attributes:
        allow_record_duplicates (RuleParameter): Configuration for record duplication rule
        allow_key_duplicates (RuleParameter): Configuration for key duplication rule
        foreign_key_checks (RuleParameter): Configuration for foreign key validation
        statistical_bounds (RuleParameter): Configuration for statistical anomaly detection
    """
    allow_record_duplicates: RuleParameter = Field(..., description="Duplicate record handling rule")
    allow_key_duplicates: RuleParameter = Field(..., description="Duplicate key handling rule")
    foreign_key_checks: Optional[RuleParameter] = Field(None, description="Foreign key check rule")
    statistical_bounds: Optional[RuleParameter] = Field(None, description="Statistical anomaly detection rule")


class DataValidationSchemaConfig(BaseModel):
    """
    Data Validation Schema Configuration Model

    Attributes:
        schema_version (str): Version of the schema
        columns (dict): Column configurations
        quality_rules (dict): Data quality rules
        metadata (Optional[dict]): Metadata information
    """
    schema_version: str = Field(..., description="Version of the schema")
    columns: Dict[str, ColumnParameter] = Field(..., description="Column configurations")
    quality_rules: QualityRulesParameter = Field(..., description="Data quality rules")
    metadata: Optional[dict] = Field(None, description="Metadata information")

In [12]:
DataValidationSchemaConfig(**schema).columns['birthdate']

ColumnParameter(description="Customer's date of birth", type='date', data_type='DATE', nullable=True, unique=None, constraints=[ConstraintParameter(type='date_range', value=None, description='Reasonable date range (assume minimum 18 years old)'), ConstraintParameter(type='not_future', value=None, description='Cannot be a future date')])

### Test

In [25]:
"""
Tests for data_validation_schema.py

Run with:
    pytest -q
"""
from datetime import datetime
import pytest
from pydantic import ValidationError


def make_sample_column():
    return {
        "description": "User identifier",
        "type": "integer",
        "data_type": "INT",
        "nullable": False,
        "unique": True,
        # constraints may be an empty list or contain ConstraintParameter objects;
        # here we provide an empty list to exercise default handling.
        "constraints": [],
    }


def make_sample_quality_rules():
    return {
        "allow_record_duplicates": {"enabled": False, "description": "No duplicate records allowed"},
        "allow_key_duplicates": {"enabled": False},
        # optional fields left out to ensure optional handling
    }


def make_valid_schema_payload():
    return {
        "schema_version": "1.0.0",
        "columns": {
            "user_id": make_sample_column(),
            "email": {
                "description": "Email address",
                "type": "string",
                "data_type": "VARCHAR(255)",
                "nullable": False,
                "unique": True,
                # test that constraints default_factory works when omitted
            },
        },
        "quality_rules": make_sample_quality_rules(),
        "metadata": {"created_by": "test-suite", "created_at": datetime.utcnow().isoformat()},
    }


def test_columnparameter_accepts_valid_data():
    col = ColumnParameter(**make_sample_column())
    assert col.description == "User identifier"
    assert col.type == "integer"
    assert col.data_type == "INT"
    assert col.nullable is False
    assert col.unique is True
    # constraints should be a list (default_factory ensures [] when omitted)
    assert isinstance(col.constraints, list)


def test_ruleparameter_requires_enabled():
    # enabled is required
    with pytest.raises(ValidationError):
        RuleParameter()  # missing required field

    # valid when enabled provided
    r = RuleParameter(enabled=True)
    assert r.enabled is True
    assert r.description is None


def test_qualityrules_accepts_required_and_optional_fields():
    payload = {
        "allow_record_duplicates": {"enabled": True},
        "allow_key_duplicates": {"enabled": False},
        # leave optional fields out to ensure they are treated as None
    }
    qr = QualityRulesParameter(**payload)
    assert isinstance(qr.allow_record_duplicates, RuleParameter)
    assert qr.foreign_key_checks is None
    assert qr.statistical_bounds is None


def test_data_validation_schema_config_roundtrip_and_types():
    payload = make_valid_schema_payload()
    cfg = DataValidationSchemaConfig(**payload)

    assert cfg.schema_version == "1.0.0"
    # columns should be parsed into a dict of ColumnParameter
    assert "user_id" in cfg.columns
    assert isinstance(cfg.columns["user_id"], ColumnParameter)
    assert cfg.columns["email"].description == "Email address"

    # quality_rules should be parsed into QualityRulesParameter
    assert isinstance(cfg.quality_rules, QualityRulesParameter)
    assert cfg.metadata["created_by"] == "test-suite"

    # serializing to dict should produce JSON-serializable primitives
    d = cfg.dict()
    assert d["schema_version"] == payload["schema_version"]
    assert "columns" in d and "user_id" in d["columns"]


def test_missing_required_fields_in_schema_config_raises():
    # missing schema_version
    payload = make_valid_schema_payload()
    payload.pop("schema_version")
    with pytest.raises(ValidationError):
        DataValidationSchemaConfig(**payload)

    # missing columns
    payload = make_valid_schema_payload()
    payload.pop("columns")
    with pytest.raises(ValidationError):
        DataValidationSchemaConfig(**payload)

    # missing quality_rules
    payload = make_valid_schema_payload()
    payload.pop("quality_rules")
    with pytest.raises(ValidationError):
        DataValidationSchemaConfig(**payload)


def test_column_defaults_for_constraints_and_nullable():
    # If constraints omitted, default_factory should give an empty list
    col_payload = {
        "description": "Test",
        "type": "string",
        "nullable": True,
    }
    # Provide minimal required fields except constraints and data_type
    # dataclass/model may require all fields; if 'type' and 'description' are sufficient, this will pass
    col = ColumnParameter(**{"description": "Test", "type": "string"})
    assert col.constraints == [] or isinstance(col.constraints, list)
    assert isinstance(col.nullable, bool)


def test_quality_rules_validation_errors_are_informative():
    # allow_record_duplicates must be present and be a RuleParameter-like mapping
    bad_payload = {
        "schema_version": "1.0",
        "columns": {"a": {"description": "x", "type": "string"}},
        "quality_rules": {"allow_record_duplicates": "not-a-dict", "allow_key_duplicates": {"enabled": True}},
    }
    with pytest.raises(ValidationError) as excinfo:
        DataValidationSchemaConfig(**bad_payload)

    # error should mention the problematic field
    msg = str(excinfo.value)
    assert "allow_record_duplicates" in msg or "quality_rules" in msg

ModuleNotFoundError: No module named 'pytest'

In [ ]:
def test_column_parameter():
    params = ColumnParameter(
        type = "enum",
        value = ["M", "F", "Other", "Unknown"],
        description = "Valid gender values"
    )
    assert params.type == "enum"
    assert params.value == ["M", "F", "Other", "Unknown"]
    assert params.description == "Valid gender values"

In [23]:
print(test_constraint_parameter())

AssertionError: 